In [4]:
import json
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
CORPUS_PATH = Path("../data/corpus.jsonl")

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")

Documentos carregados: 2000


In [6]:
texts = [
    d["title"] + " " + d["abstract"]
    for d in docs
]

vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True
)

X = vectorizer.fit_transform(texts)

print("Documentos:", X.shape[0])
print("Termos:", X.shape[1])

Documentos: 2000
Termos: 18974


In [7]:
def search(query, k=10):
    q_vec = vectorizer.transform([query])

    scores = cosine_similarity(q_vec, X).flatten()

    idxs = scores.argsort()[::-1][:k]

    return [
        (idx, scores[idx], docs[idx])
        for idx in idxs
    ]

In [8]:
query = "maritime target detection"

results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:.4f}] {d['title']}")

 1. [0.3221] Multi-Field Hybrid Retrieval-Augmented Generation for Maritime Accident Root Cause Analysis
 2. [0.2211] HARBOR: Heading Analysis and Reconstruction from Behavioral Observation and Radar
 3. [0.1855] Rarity-Gated Context Conditioning for Offline Imitation Learning-Based Maritime Anomaly Detection
 4. [0.1790] Consensus-based Agentic Large Language Model Framework for Harmonized Tariff Schedule Code Classification
 5. [0.1690] A Unifying Lens on Supervised Fine-Tuning Through Target Distribution Design
 6. [0.1602] A Source Domain is All You Need: Source-Only Cross-OS Transfer Learning for APT Anomaly Detection via Semantic Alignment and Optimal Transport
 7. [0.1432] Decoupled Motion Representation Learning for Moving Infrared Small Target Detection
 8. [0.1419] Geometrically Averaged Hard Target Updates for Linear Q-Learning
 9. [0.1342] A Turbo-Inference Strategy for Object Detection and Instance Segmentation
10. [0.1338] Adversarial Attack and Disturbance Detection by H

In [9]:
queries_path = Path("../eval/queries.tsv")

queries = pd.read_csv(
    queries_path,
    sep="\t",
    names=["qid", "text"]
)

print(queries)

   qid                                           text
0   Q1                      maritime target detection
1   Q2                      maritime object detection
2   Q3       human detection in maritime environments
3   Q4         person detection for search and rescue
4   Q5                  small object detection at sea
5   Q6             infrared maritime target detection
6   Q7      computer vision for maritime surveillance
7   Q8  deep learning for maritime target recognition
8   Q9  artificial intelligence for search and rescue
9  Q10      maritime surveillance using EO IR sensors


In [10]:
runs_dir = Path("./runs")
runs_dir.mkdir(exist_ok=True)

run_path = runs_dir / "knn.trec"

with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():

        results = search(q["text"], k=100)

        for rank, (idx, score, d) in enumerate(results, 1):
            f.write(
                f"{q['qid']} Q0 {d['arxiv_id']} "
                f"{rank} {score:.6f} knn\n"
            )

print("Run salva em:", run_path)

Run salva em: runs\knn.trec


In [11]:
with open(run_path, "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline().strip())

Q1 Q0 2606.13249 1 0.322115 knn
Q1 Q0 2606.14006 2 0.221087 knn
Q1 Q0 2606.13311 3 0.185515 knn
Q1 Q0 2606.16987 4 0.178991 knn
Q1 Q0 2606.11189 5 0.168956 knn


In [12]:
ids = [
    "2606.13311",
    "2606.14006",
    "2606.13249",
    "2606.16987",
    "2606.15286",
    "2606.10216",
    "2606.15409",
    "2606.12673",
    "2606.14475",
    "2606.17478",
    "2606.11189",
    "2606.10835",
    "2606.12371",
    "2606.09536"
]

for d in docs:
    if d["arxiv_id"] in ids:
        print("=" * 80)
        print(d["arxiv_id"])
        print(d["title"])

2606.09536
Adversarial Attack and Disturbance Detection by Hadamard-Coded Output Representations for Object Detection and Semantic Segmentation
2606.10216
A Source Domain is All You Need: Source-Only Cross-OS Transfer Learning for APT Anomaly Detection via Semantic Alignment and Optimal Transport
2606.10835
Geometrically Averaged Hard Target Updates for Linear Q-Learning
2606.11189
A Unifying Lens on Supervised Fine-Tuning Through Target Distribution Design
2606.12371
A Turbo-Inference Strategy for Object Detection and Instance Segmentation
2606.12673
A Zero-shot Generalized Graph Anomaly Detection Framework via Node Reconstruction
2606.13249
Multi-Field Hybrid Retrieval-Augmented Generation for Maritime Accident Root Cause Analysis
2606.13311
Rarity-Gated Context Conditioning for Offline Imitation Learning-Based Maritime Anomaly Detection
2606.14006
HARBOR: Heading Analysis and Reconstruction from Behavioral Observation and Radar
2606.14475
Value-order Decomposition for Generalist Ano

In [13]:
for q in queries["text"]:
    print("=" * 80)
    print(q)

    results = search(q, k=10)

    for rank, (idx, score, d) in enumerate(results, 1):
        print(f"{rank:2d} - {d['arxiv_id']} - {d['title']}")

maritime target detection
 1 - 2606.13249 - Multi-Field Hybrid Retrieval-Augmented Generation for Maritime Accident Root Cause Analysis
 2 - 2606.14006 - HARBOR: Heading Analysis and Reconstruction from Behavioral Observation and Radar
 3 - 2606.13311 - Rarity-Gated Context Conditioning for Offline Imitation Learning-Based Maritime Anomaly Detection
 4 - 2606.16987 - Consensus-based Agentic Large Language Model Framework for Harmonized Tariff Schedule Code Classification
 5 - 2606.11189 - A Unifying Lens on Supervised Fine-Tuning Through Target Distribution Design
 6 - 2606.10216 - A Source Domain is All You Need: Source-Only Cross-OS Transfer Learning for APT Anomaly Detection via Semantic Alignment and Optimal Transport
 7 - 2606.15286 - Decoupled Motion Representation Learning for Moving Infrared Small Target Detection
 8 - 2606.10835 - Geometrically Averaged Hard Target Updates for Linear Q-Learning
 9 - 2606.12371 - A Turbo-Inference Strategy for Object Detection and Instance Segm